***

# **Mortgage Lending Data**

***

In [3]:
# packages

import pandas as pd

In [62]:
# Post 2018 data

df = pd.read_csv('https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=06101,06115,06113,06061,06017,06067&years=2023')

C:\Users\jchoy\AppData\Local\Temp\ipykernel_91932\1553158523.py:3: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=06101,06115,06113,06061,06017,06067&years=2023')


In [25]:
df = df[df['action_taken'].isin([1, 3])]

In [41]:
display(df['derived_ethnicity'].value_counts())

display(df['derived_race'].value_counts())

display(df['derived_sex'].value_counts())


derived_ethnicity
Not Hispanic or Latino     40922
Ethnicity Not Available    22334
Hispanic or Latino          8410
Joint                       2799
Free Form Text Only           61
Name: count, dtype: int64

derived_race
White                                        32123
Race Not Available                           24014
Asian                                        11032
Black or African American                     3228
Joint                                         2522
American Indian or Alaska Native               656
Native Hawaiian or Other Pacific Islander      596
2 or more minority races                       328
Free Form Text Only                             27
Name: count, dtype: int64

derived_sex
Joint                24583
Male                 20132
Female               15315
Sex Not Available    14496
Name: count, dtype: int64

In [42]:
df.groupby(['county_code', 'derived_ethnicity', 'derived_race', 'derived_sex', 'action_taken']).size().reset_index(name='count')

,county_code,derived_ethnicity,derived_race,derived_sex,action_taken,count
0,6017,Ethnicity Not Available,American Indian or Alaska Native,Female,3,1
1,6017,Ethnicity Not Available,American Indian or Alaska Native,Male,1,3
2,6017,Ethnicity Not Available,Asian,Female,1,2
3,6017,Ethnicity Not Available,Asian,Female,3,1
4,6017,Ethnicity Not Available,Asian,Female,4,1
...,...,...,...,...,...,...
1994,6115,Not Hispanic or Latino,White,Male,6,6
1995,6115,Not Hispanic or Latino,White,Male,8,2
1996,6115,Not Hispanic or Latino,White,Sex Not Available,1,5
1997,6115,Not Hispanic or Latino,White,Sex Not Available,3,6


In [43]:
df['demographic_value'] = df.apply(lambda x: x['derived_sex'] if x['derived_sex'] == 'Female' else x['derived_race'], axis=1)

df = df[df['loan_purpose'].isin([1, 2, 31])]

# Group by county, loan purpose, and demographic value, and sum conditions for originated and denied
result = df.groupby(['county_code', 'loan_purpose', 'demographic_value']).agg(
    originated=('action_taken', lambda x: (x == 1).sum()),
    denied=('action_taken', lambda x: (x == 3).sum())
).reset_index()

result = result[result['demographic_value'].isin()]

TypeError: Series.isin() missing 1 required positional argument: 'values'

In [63]:
df = df[df['loan_purpose'].isin([1, 2, 31])]

def get_demographics(row):
    demographics = []
    # Add race/ethnicity
    if row['derived_race'] in ['Black or African American', 'White', 'Asian']:
        demographics.append(row['derived_race'])
    else:
        demographics.append('Non-White')
    # Add gender
    if row['derived_sex'] in ['Male', 'Female']:
        demographics.append(row['derived_sex'])
    # Latino 
    if row['derived_ethnicity'] in ['Not Hispanic or Latino', 'Hispanic or Latino']:
        demographics.append(row['derived_ethnicity'])
    return demographics

# Apply the function to create a list of demographics for each row
df['demographics'] = df.apply(get_demographics, axis=1)

# Explode rows so each demographic gets counted individually
df_exploded = df.explode('demographics')

# Group by county and demographic to count occurrences
result = df_exploded.groupby(['county_code', 'demographics']).size().reset_index(name='count')


In [66]:
df['action_taken'].value_counts()

action_taken
1    30053
4     7366
6     7091
3     6699
5     1768
2     1486
8      203
7       28
Name: count, dtype: int64

In [91]:

def demos(row):
    # if row['derived_sex'] in ['Female', 'Male']:
    #     return row['derived_sex']  # Prioritize gender if Male or Female
    if row['derived_ethnicity'] == 'Hispanic or Latino':
        return 'Latino'
    elif row['derived_ethnicity'] == 'Not Hispanic or Latino':
        return 'Non-Latino'
    elif row['derived_race'] in ['Black or African American', 'White', 'Asian']:
        return row['derived_race']
    else:
        return 'Non-White'  

    
df_filtered = df[df['loan_purpose'].isin([1, 2, 31])]
df_filtered = df[df['action_taken'].isin([1, 3])]
df_filtered['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
df_filtered['demographics'] = df_filtered.apply(demos, axis=1)


df_exploded = df_filtered.explode('demographics')


df_exploded['originated'] = df_exploded['action_taken'].apply(lambda x: 1 if x == 1 else 0)
df_exploded['denied'] = df_exploded['action_taken'].apply(lambda x: 1 if x == 3 else 0)

result = df_exploded.groupby(['activity_year', 'county_code', 'demographics', 'loan_purpose']).agg(
    originated=('originated', 'sum'),
    denied=('denied', 'sum')
).reset_index()

aggregated = result.groupby(['activity_year', 'county_code', 'demographics']).agg(
    total_originated=('originated', 'sum'),
    total_denied=('denied', 'sum')
).reset_index()

aggregated['loan_purpose'] = 'All'

df_with_totals = pd.concat([result, aggregated[['activity_year', 'county_code', 'loan_purpose', 'demographics', 'total_originated', 'total_denied']]], ignore_index=True)
df_with_totals['originated'] = df_with_totals['originated'].fillna(df_with_totals['total_originated'])
df_with_totals['denied'] = df_with_totals['denied'].fillna(df_with_totals['total_denied'])
df_with_totals.drop(columns=['total_originated', 'total_denied'], inplace=True)

df_with_totals['county_name'] = df_with_totals['county_code'].map({6101: 'Sutter', 6115: 'Yuba', 6113: 'Yolo', 6061: 'Placer', 6017: 'El Dorado', 6067: 'Sacramento'})

C:\Users\jchoy\AppData\Local\Temp\ipykernel_91932\3907832444.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
C:\Users\jchoy\AppData\Local\Temp\ipykernel_91932\3907832444.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['demographics'] = df_filtered.apply(demos, axis=1)


In [95]:
df_with_totals

,activity_year,county_code,demographics,loan_purpose,originated,denied,county_name
0,2023,6017,Asian,Home improvement,3.0,2.0,El Dorado
1,2023,6017,Asian,Home purchase,4.0,2.0,El Dorado
2,2023,6017,Black or African American,Home purchase,1.0,0.0,El Dorado
3,2023,6017,Latino,Home improvement,30.0,23.0,El Dorado
4,2023,6017,Latino,Home purchase,118.0,15.0,El Dorado
...,...,...,...,...,...,...,...
131,2023,6115,Black or African American,All,6.0,4.0,Yuba
132,2023,6115,Latino,All,237.0,87.0,Yuba
133,2023,6115,Non-Latino,All,693.0,165.0,Yuba
134,2023,6115,Non-White,All,223.0,59.0,Yuba


In [ ]:
# let's make a function that will clean and shape our data the same way the Delaware guys did it
https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=06101,06115,06113,06061,06017,06067&years=2023

def hmda_maker(years, counties):

    dfs = []

    for year in years:
        url = 'https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=' + ','.join(counties) + '&years=' + year
        
        df = pd.read_csv(url)

        df = df[df['loan_purpose'].isin([1, 2, 31])]
        df = df[df['action_taken'].isin([1, 3])]
        df['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
        df['demographics'] = df.apply(demos, axis=1)

        df_exploud = df.explode('demographics')

        df_exploud['originations'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 1 else 0)
        df_exploud['denials'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 3 else 0)

        df_initial = df_exploud.groupby(['activity_year', 'county_code', 'demographics', 'loan_purpose']).agg(
            originated=('originations', 'sum'),
            denied=('denials', 'sum')
        ).reset_index()

        df_sums = result.groupby(['activity_year', 'county_code', 'demographics']).agg(
            total_originated=('originated', 'sum'),
            total_denied=('denied', 'sum')
        ).reset_index()

        df_sums['loan_purpose'] = 'All'

        df_final = pd.concat([df_initial, df_sums[['activity_year', 'county_code', 'loan_purpose', 'demographics', 'total_originated', 'total_denied']]], ignore_index=True)
        df_final['originations'] = df_final['originated'].fillna(df_final['total_originated'])
        df_final['denials'] = df_final['denied'].fillna(df_final['total_denied'])
        df_final.drop(columns=['total_originated', 'total_denied'], inplace=True)
        df_final['county_name'] = df_final['county_code'].map({6101: 'Sutter', 6115: 'Yuba', 6113: 'Yolo', 6061: 'Placer', 6017: 'El Dorado', 6067: 'Sacramento'})

        dfs.append(df_final)
    
    df_county = pd.concat(dfs, ignore_index=True)
    df_county['total'] = df_county['originations'] + df_county['denials']
    df_county['denial_rate'] = df_county['denials'] / df_county['total']
    df_county['origination_rate'] = df_county['originations'] / df_county['total']
    
    return df_county

,county_code,demographics,loan_purpose,originated_x,denied_x,originated_y,denied_y
0,6017,Asian,Home improvement,26,9,155,33
1,6017,Asian,Home purchase,124,19,155,33
2,6017,Asian,Refinancing,5,5,155,33
3,6017,Black or African American,Home improvement,1,4,13,7
4,6017,Black or African American,Home purchase,11,3,13,7
5,6017,Black or African American,Refinancing,1,0,13,7
6,6017,Female,Home improvement,93,61,460,133
7,6017,Female,Home purchase,334,51,460,133
8,6017,Female,Refinancing,33,21,460,133
9,6017,Hispanic or Latino,Home improvement,30,23,159,42


In [47]:
result['demographic_category']

0                          Asian
1      Black or African American
2                         Female
3                         Latino
4                           Male
                 ...            
126                       Latino
127                         Male
128                   Non-Latino
129                    Non-White
130                        White
Name: demographic_category, Length: 131, dtype: object

In [52]:
result[result['county_code'] == 6017]

,county_code,loan_purpose,demographic_category,originated,denied
0,6017,1,Asian,4,2
1,6017,1,Black or African American,1,0
2,6017,1,Latino,118,15
3,6017,1,Non-Latino,1355,166
4,6017,1,Non-White,415,55
5,6017,1,White,145,10
6,6017,2,Asian,3,2
7,6017,2,Latino,30,23
8,6017,2,Non-Latino,391,190
9,6017,2,Non-White,78,49


In [56]:
def categorize_demographic(row):
    if row['derived_sex'] in ['Female', 'Male']:
        return row['derived_sex']  # Prioritize gender if specified as Male or Female
    elif row['derived_ethnicity'] == 'Latino':
        return 'Latino'
    elif row['derived_ethnicity'] == 'Non-Latino':
        return 'Non-Latino'
    elif row['derived_race'] in ['Black', 'White', 'Asian']:
        return row['derived_race']
    else:
        return 'Non-White'  # Assume other races as "Non-White"

df['demographic_category'] = df.apply(categorize_demographic, axis=1)


# Step 1: Group by the relevant columns and sum originated and denied counts
summary_df = df.groupby(['county_code', 'loan_purpose', 'demographic_category']).agg(
    total_originated=('originated', 'sum'),
    total_denied=('denied', 'sum')
).reset_index()

# Step 2: Calculate the total for each row and the ratios
summary_df['total'] = summary_df['total_originated'] + summary_df['total_denied']
summary_df['origination_ratio'] = summary_df['total_originated'] / summary_df['total']
summary_df['denial_ratio'] = summary_df['total_denied'] / summary_df['total']

# Step 3: Add an "All" row for each demographic category, summing across loan purposes
all_purposes_summary = summary_df.groupby(['Year', 'county_code', 'County', 'demographic_category']).agg(
    total_originated=('total_originated', 'sum'),
    total_denied=('total_denied', 'sum')
).reset_index()

all_purposes_summary['loan_purpose'] = 'All'  # Add 'All' to represent the aggregate row
all_purposes_summary['total'] = all_purposes_summary['total_originated'] + all_purposes_summary['total_denied']
all_purposes_summary['origination_ratio'] = all_purposes_summary['total_originated'] / all_purposes_summary['total']
all_purposes_summary['denial_ratio'] = all_purposes_summary['total_denied'] / all_purposes_summary['total']

# Step 4: Append the "All" rows to the original summary dataframe
final_summary = pd.concat([summary_df, all_purposes_summary], ignore_index=True)

# Sort for better readability if desired
final_summary = final_summary.sort_values(by=['Year', 'county_code', 'County', 'demographic_category', 'loan_purpose'])

print(final_summary)

KeyError: "Column(s) ['denied', 'originated'] do not exist"

In [54]:
final_result

,county_code,loan_purpose,demographic_category,originated,denied
0,6017,1,Asian,65,5
1,6017,1,Female,334,51
2,6017,1,Male,517,80
3,6017,1,Non-White,424,53
4,6017,1,White,698,59
...,...,...,...,...,...
121,6115,1,Male,348,62
122,6115,2,Female,20,26
123,6115,2,Male,33,32
124,6115,31,Female,17,8
